In [1]:
%env CUDA_VISIBLE_DEVICES=4

from diffusers import DiffusionPipeline
from transformers import CLIPTextModel, CLIPTokenizer, BlipForConditionalGeneration, BlipProcessor, AutoModelForCausalLM, AutoTokenizer

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from IPython.display import display
from PIL import Image

import pyiqa
from torchmetrics.multimodal import CLIPScore

env: CUDA_VISIBLE_DEVICES=4


/home/anmilka/miniconda3/envs/common/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def adjust_embeddings(embeddings, target_dim=768):
    current_dim = embeddings.shape[-1]
    if target_dim - current_dim > 0:
        embeddings = F.pad(embeddings, (0, target_dim - current_dim), "constant", 0)
    else:
        embeddings = embeddings[:, :, :target_dim]
    return embeddings

def evaluate_image_quality(image: Image.Image, prompt: str, device: str):
    transform_to_tensor_float = transforms.ToTensor()
    image_tensor_float = transform_to_tensor_float(image).unsqueeze(0).to(device)
    
    transform_to_tensor_int = transforms.PILToTensor()
    image_tensor_int = transform_to_tensor_int(image).unsqueeze(0).to(device)
    scores = {}

    # соответствие тексту
    clip_score_metric = CLIPScore().to(device)
    score = clip_score_metric(image_tensor_int, [prompt])
    scores['CLIP Score'] = score.detach().item()

    niqe_metric = pyiqa.create_metric('niqe', device=device)
    brisque_metric = pyiqa.create_metric('brisque', device=device)
    piqe_metric = pyiqa.create_metric('piqe', device=device)
    clipiqa_metric = pyiqa.create_metric('clipiqa', device=device)

    # естественность
    score = niqe_metric(image_tensor_float)
    scores['NIQE'] = score.item()
    
    # отсутствие искажений
    score = brisque_metric(image_tensor_float)
    scores['BRISQUE'] = score.item()
    
    # воспринимаемое качество
    score = piqe_metric(image_tensor_float)
    scores['PIQE'] = score.item()
    
    # общее качество
    score = clipiqa_metric(image_tensor_float)
    scores['CLIPIQA'] = score.item()

    return scores

def infer_model(pipe, text_encoder, tokenizer, prompt, negative_prompt="", adapter=None):
    for_qwen = True if 'qwen' in text_encoder.config.model_type else False
    for_blip = True if 'blip' in text_encoder.config.model_type else False

    prompt_tokens = tokenizer(
        text=prompt, 
        padding="max_length", 
        max_length=77, 
        truncation=True, 
        return_tensors="pt"
    )

    negative_prompt_tokens = tokenizer(
        text=negative_prompt, 
        padding="max_length", 
        max_length=77, 
        truncation=True, 
        return_tensors="pt"
    )

    prompt_input_ids = prompt_tokens.input_ids.to(pipe.device)
    negative_prompt_input_ids = negative_prompt_tokens.input_ids.to(pipe.device)

    with torch.no_grad():
        prompt_embeds = text_encoder(prompt_input_ids, output_hidden_states=True) if (for_qwen or for_blip) else text_encoder(prompt_input_ids)[0]
        negative_prompt_embeds = text_encoder(negative_prompt_input_ids, output_hidden_states=True) if (for_qwen or for_blip) else text_encoder(negative_prompt_input_ids)[0]

    prompt_embeds = prompt_embeds.hidden_states[-1] if (for_qwen or for_blip) else prompt_embeds
    negative_prompt_embeds = negative_prompt_embeds.hidden_states[-1] if (for_qwen or for_blip) else negative_prompt_embeds
    
    prompt_embeds = adjust_embeddings(prompt_embeds) if adapter is None else adapter(prompt_embeds)
    negative_prompt_embeds = adjust_embeddings(negative_prompt_embeds)if adapter is None else adapter(negative_prompt_embeds)

    images = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        num_inference_steps=50
    ).images

    display(images[0])

    try:
        quality_scores = evaluate_image_quality(
            image=images[0],
            prompt=prompt,
            device=str(pipe.device)
        )

        for metric, score in quality_scores.items():
            print(f"{metric:<30}: {score:.4f}")
    except:
        return

prompts = [
    "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k",
    "Marble dachshund with blue eyes, detailed, 8k",
    "A girl in pink dress with a bouquet of roses, detailed, 8k",
    "Lilo and Stitch, detailed, 8k",
    "The starry sky, detailed, 8k"
]

device = 'cuda' if torch.cuda.is_available() else "cpu"
pipe = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5").to(device)

Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  5.41it/s]


In [3]:
clip_text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")

for prompt in prompts:
    infer_model(pipe, clip_text_encoder, clip_tokenizer, prompt)

del clip_text_encoder
torch.cuda.empty_cache()

100%|██████████| 50/50 [00:08<00:00,  6.14it/s]


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIP Score                    : 32.6265
NIQE                          : 4.3923
BRISQUE                       : 8.2650
PIQE                          : 17.5228
CLIPIQA                       : 0.8016


100%|██████████| 50/50 [00:07<00:00,  6.68it/s]


CLIP Score                    : 33.3433
NIQE                          : 3.4641
BRISQUE                       : 0.0716
PIQE                          : 22.2134
CLIPIQA                       : 0.8434


...

CLIP Score                    : 27.8935
NIQE                          : 4.3337
BRISQUE                       : -0.8509
PIQE                          : 12.1682
CLIPIQA                       : 0.7039


100%|██████████| 50/50 [00:07<00:00,  6.66it/s]


CLIP Score                    : 23.6335
NIQE                          : 4.5542
BRISQUE                       : -6.2745
PIQE                          : 24.9317
CLIPIQA                       : 0.7942


100%|██████████| 50/50 [00:07<00:00,  6.68it/s]


CLIP Score                    : 20.8113
NIQE                          : 6.4077
BRISQUE                       : 5.7817
PIQE                          : 33.9241
CLIPIQA                       : 0.6775


In [4]:
clip_base_text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_base_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

for prompt in prompts:
    infer_model(pipe, clip_base_text_encoder, clip_base_tokenizer, prompt)

del clip_base_text_encoder
torch.cuda.empty_cache()

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:07<00:00,  6.67it/s]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


/home/anmilka/miniconda3/envs/common/lib/python3.10/site-packages/pyiqa/matlab_utils/functions.py:173: UserWarning: cov(): degrees of freedom is <= 0. Correction should be strictly less than the number of observations. (Triggered internally at /pytorch/aten/src/ATen/native/Correlation.cpp:116.)
  return torch.cov(tensor, correction=correction)
/home/anmilka/miniconda3/envs/common/lib/python3.10/site-packages/pyiqa/archs/niqe_arch.py:161: UserWarning: torch.linalg.svd: During SVD computation with the selected cusolver driver, batches 0 failed to converge. A more accurate method will be used to compute the SVD as a fallback. Check doc at https://pytorch.org/docs/stable/generated/torch.linalg.svd.html (Triggered internally at /pytorch/aten/src/ATen/native/cuda/linalg/BatchLinearAlgebraLib.cpp:702.)
  invcov_param = torch.linalg.pinv((cov_pris_param + cov_distparam) / 2)
100%|██████████| 50/50 [00:07<00:00,  6.61it/s]


CLIP Score                    : 14.0200
NIQE                          : 8.0142
BRISQUE                       : 68.6227
PIQE                          : 43.4533
CLIPIQA                       : 0.3155


100%|██████████| 50/50 [00:07<00:00,  6.69it/s]


CLIP Score                    : 8.3361
NIQE                          : 69.6225
BRISQUE                       : 153.5910
PIQE                          : 31.2134
CLIPIQA                       : 0.7653


100%|██████████| 50/50 [00:07<00:00,  6.67it/s]


CLIP Score                    : 16.8394
NIQE                          : 7.0358
BRISQUE                       : 55.8035
PIQE                          : 55.6894
CLIPIQA                       : 0.3556


100%|██████████| 50/50 [00:07<00:00,  6.66it/s]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


In [5]:
blip_text_encoder = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device).text_decoder
blip_tokenizer = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")

for prompt in prompts:
    infer_model(pipe, blip_text_encoder, blip_tokenizer, prompt)

del blip_text_encoder
torch.cuda.empty_cache()

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
100%|██████████| 50/50 [00:07<00:00,  6.67it/s]


CLIP Score                    : 12.2048
NIQE                          : 4.2766
BRISQUE                       : 15.7392
PIQE                          : 26.1560
CLIPIQA                       : 0.5022


100%|██████████| 50/50 [00:07<00:00,  6.38it/s]


CLIP Score                    : 7.2699
NIQE                          : 2.5287
BRISQUE                       : 9.2748
PIQE                          : 18.5617
CLIPIQA                       : 0.5134


100%|██████████| 50/50 [00:07<00:00,  6.67it/s]


CLIP Score                    : 12.1550
NIQE                          : 6.1231
BRISQUE                       : 18.2095
PIQE                          : 50.7706
CLIPIQA                       : 0.8580


100%|██████████| 50/50 [00:07<00:00,  6.68it/s]


CLIP Score                    : 16.8998
NIQE                          : 4.8434
BRISQUE                       : 34.0043
PIQE                          : 55.3419
CLIPIQA                       : 0.7937


100%|██████████| 50/50 [00:07<00:00,  6.68it/s]


CLIP Score                    : 11.9060
NIQE                          : 5.4426
BRISQUE                       : 45.7253
PIQE                          : 48.0934
CLIPIQA                       : 0.4862


In [3]:
qwen_text_encoder = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Instruct-2507",
    torch_dtype=torch.float32,
    trust_remote_code=True
).to(device)
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507", trust_remote_code=True)

for prompt in prompts:
    infer_model(pipe, qwen_text_encoder, qwen_tokenizer, prompt)

del qwen_text_encoder
torch.cuda.empty_cache()

`torch_dtype` is deprecated! Use `dtype` instead!
100%|██████████| 50/50 [00:08<00:00,  6.23it/s]


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIP Score                    : 14.8795
NIQE                          : 6.4744
BRISQUE                       : 39.1953
PIQE                          : 47.0179
CLIPIQA                       : 0.1966


100%|██████████| 50/50 [00:07<00:00,  6.66it/s]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


/home/anmilka/miniconda3/envs/common/lib/python3.10/site-packages/pyiqa/matlab_utils/functions.py:173: UserWarning: cov(): degrees of freedom is <= 0. Correction should be strictly less than the number of observations. (Triggered internally at /pytorch/aten/src/ATen/native/Correlation.cpp:116.)
  return torch.cov(tensor, correction=correction)
/home/anmilka/miniconda3/envs/common/lib/python3.10/site-packages/pyiqa/archs/niqe_arch.py:161: UserWarning: torch.linalg.svd: During SVD computation with the selected cusolver driver, batches 0 failed to converge. A more accurate method will be used to compute the SVD as a fallback. Check doc at https://pytorch.org/docs/stable/generated/torch.linalg.svd.html (Triggered internally at /pytorch/aten/src/ATen/native/cuda/linalg/BatchLinearAlgebraLib.cpp:702.)
  invcov_param = torch.linalg.pinv((cov_pris_param + cov_distparam) / 2)
100%|██████████| 50/50 [00:07<00:00,  6.68it/s]


CLIP Score                    : 19.4027
NIQE                          : 7.5295
BRISQUE                       : 45.5612
PIQE                          : 72.5083
CLIPIQA                       : 0.1484


100%|██████████| 50/50 [00:07<00:00,  6.68it/s]


CLIP Score                    : 16.0723
NIQE                          : 8.9746
BRISQUE                       : 64.9539
PIQE                          : 100.0000
CLIPIQA                       : 0.3225


100%|██████████| 50/50 [00:07<00:00,  6.67it/s]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


In [ ]:
%env CUDA_VISIBLE_DEVICES=4

from datasets import load_dataset
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
from diffusers import DiffusionPipeline
from transformers import CLIPTextModel, CLIPTokenizer, BlipForConditionalGeneration, BlipProcessor, AutoModelForCausalLM, AutoTokenizer

import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

device = 'cuda:0' if torch.cuda.is_available() else "cpu"


ds = load_dataset("Gustavosta/Stable-Diffusion-Prompts")

sample_size = 40000
prompts_sample = ds['train']['Prompt'][:sample_size]

def extract_embeddings(text_encoder, tokenizer, prompts, batch_size=32, max_length=77):
    for_qwen = True if 'qwen' in text_encoder.config.model_type else False
    for_blip = True if 'blip' in text_encoder.config.model_type else False

    embeddings = []
    
    for i in tqdm(range(0, len(prompts), batch_size), desc="Извлечение эмбеддингов"):
        batch_prompts = prompts[i:i+batch_size]
        tokens = tokenizer(
            text=batch_prompts,
            padding="max_length",
            max_length=max_length,
            truncation=True,
            return_tensors="pt"
        )
        input_ids = tokens.input_ids.to(device)

        with torch.no_grad():
            batch_embeddings = text_encoder(input_ids, output_hidden_states=True) if (for_qwen or for_blip) else text_encoder(input_ids)[0]

        batch_embeddings = batch_embeddings.hidden_states[-1] if (for_qwen or for_blip) else batch_embeddings
        embeddings.append(batch_embeddings.cpu())
    
    return torch.cat(embeddings, dim=0)

env: CUDA_VISIBLE_DEVICES=4


/home/anmilka/miniconda3/envs/common/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
clip_text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
clip_embeddings = extract_embeddings(clip_text_encoder, clip_tokenizer, prompts_sample)

del clip_text_encoder
torch.cuda.empty_cache()

# clip_base_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
# clip_base_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
# clip_base_embeddings = extract_embeddings(clip_base_encoder, clip_base_tokenizer, prompts_sample)

# del clip_base_encoder
# torch.cuda.empty_cache()

# blip_encoder = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device).text_decoder
# blip_tokenizer = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
# blip_embeddings = extract_embeddings(blip_encoder, blip_tokenizer, prompts_sample)

# del blip_encoder
# torch.cuda.empty_cache()

qwen_encoder = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Instruct-2507",
    torch_dtype=torch.float32,
    trust_remote_code=True
).to(device)
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507", trust_remote_code=True)
qwen_embeddings = extract_embeddings(qwen_encoder, qwen_tokenizer, prompts_sample)

del qwen_encoder
torch.cuda.empty_cache()

Извлечение эмбеддингов: 100%|██████████| 1250/1250 [01:36<00:00, 12.93it/s]
`torch_dtype` is deprecated! Use `dtype` instead!
Извлечение эмбеддингов: 100%|██████████| 1250/1250 [35:54<00:00,  1.72s/it]


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm


class Adapter(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=3072):
        super().__init__()
        self.l1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.l2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.l1(x)
        x = self.act(x)
        x = self.l2(x)
        return x

def train_adapter(train_input, train_target, val_input, val_target, model_name, epochs=30, lr=2e-3, batch_size=2048):    
    input_dim = train_input.shape[-1]
    output_dim = train_target.shape[-1]

    adapter = Adapter(input_dim, output_dim).to(device)

    train_dataset = TensorDataset(train_input, train_target)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    val_dataset = TensorDataset(val_input, val_target)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    optimizer = torch.optim.Adam(adapter.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[20, 30], gamma=0.5)
    
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    
    for epoch in tqdm(range(epochs), desc=f"Обучение адаптера {model_name}"):
        adapter.train()
        train_epoch_loss = 0
        for batch_input, batch_target in train_dataloader:
            batch_input = batch_input.to(device)
            batch_target = batch_target.to(device)
            
            optimizer.zero_grad()
            
            output = adapter(batch_input)
            loss = criterion(output, batch_target)
            
            loss.backward()
            optimizer.step()
            
            train_epoch_loss += loss.item()
        
        adapter.eval()
        val_epoch_loss = 0
        with torch.no_grad():
            for batch_input, batch_target in val_dataloader:
                batch_input = batch_input.to(device)
                batch_target = batch_target.to(device)
                
                output = adapter(batch_input)
                loss = criterion(output, batch_target)
                val_epoch_loss += loss.item()
        
        avg_train_loss = train_epoch_loss / len(train_dataloader)
        avg_val_loss = val_epoch_loss / len(val_dataloader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        if (epoch + 1) % 5 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Эпоха {epoch+1}/{epochs}, LR: {current_lr:.6f}, Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")
        scheduler.step()
    
    return adapter, train_losses, val_losses


In [ ]:
training_losses = {}
validation_losses = {}

train_size = int(0.95 * len(prompts_sample))
val_size = len(prompts_sample) - train_size

print(f"Размер обучающей выборки: {train_size}")
print(f"Размер валидационной выборки: {val_size}")

clip_train = clip_embeddings[:train_size]
clip_val = clip_embeddings[train_size:]


print("\n=== Обучение адаптера CLIP Base -> CLIP Large ===")
clip_base_train = clip_base_embeddings[:train_size]
clip_base_val = clip_base_embeddings[train_size:]

adapter_clip_base, train_losses_clip_base, val_losses_clip_base = train_adapter(
    clip_base_train, clip_train, clip_base_val, clip_val, "CLIP_Base", epochs=30
)
torch.save(adapter_clip_base, "/home/anmilka/different/genai/adapter_clip_base.pth")


print("\n=== Обучение адаптера BLIP -> CLIP Large ===")
blip_train = blip_embeddings[:train_size]
blip_val = blip_embeddings[train_size:]

adapter_blip, train_losses_blip, val_losses_blip = train_adapter(
    blip_train, clip_train, blip_val, clip_val, "BLIP", epochs=40
)
torch.save(adapter_blip, "/home/anmilka/different/genai/adapter_blip.pth")


print("\n=== Обучение адаптера Qwen -> CLIP Large ===")
qwen_train = qwen_embeddings[:train_size]
qwen_val = qwen_embeddings[train_size:]

adapter_qwen, train_losses_qwen, val_losses_qwen = train_adapter(
    qwen_train, clip_train, qwen_val, clip_val, "Qwen", epochs=10
)
torch.save(adapter_qwen, "/home/anmilka/different/genai/adapter_qwen.pth")


print("\nОбучение всех адаптеров завершено!")

Размер обучающей выборки: 38000
Размер валидационной выборки: 2000

=== Обучение адаптера Qwen -> CLIP Large ===


Обучение адаптера Qwen:  50%|█████     | 5/10 [24:30<23:37, 283.58s/it]

Эпоха 5/10, LR: 0.002000, Train Loss: 0.710643, Val Loss: 0.708731


Обучение адаптера Qwen: 100%|██████████| 10/10 [50:17<00:00, 301.75s/it]

Эпоха 10/10, LR: 0.002000, Train Loss: 0.686448, Val Loss: 0.685901

Обучение всех адаптеров завершено!


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].plot(train_losses_clip_base, label='Train Loss', alpha=0.8, color='blue')
axes[0].plot(val_losses_clip_base, label='Val Loss', alpha=0.8, color='red')
axes[0].set_title('CLIP Base -> CLIP Large')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_losses_blip, label='Train Loss', alpha=0.8, color='blue')
axes[1].plot(val_losses_blip, label='Val Loss', alpha=0.8, color='red')
axes[1].set_title('BLIP -> CLIP Large')
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('MSE Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(train_losses_qwen, label='Train Loss', alpha=0.8, color='blue')
axes[2].plot(val_losses_qwen, label='Val Loss', alpha=0.8, color='red')
axes[2].set_title('Qwen -> CLIP Large')
axes[2].set_xlabel('Эпоха')
axes[2].set_ylabel('MSE Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [6]:
clip_base_text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_base_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
adapter_clip_base = torch.load('/home/anmilka/different/genai/adapter_clip_base.pth', weights_only=False)

for prompt in prompts:
    infer_model(pipe, clip_base_text_encoder, clip_base_tokenizer, prompt, " ", adapter_clip_base)

del clip_base_text_encoder
torch.cuda.empty_cache()

100%|██████████| 50/50 [00:07<00:00,  6.63it/s]


CLIP Score                    : 20.7734
NIQE                          : 3.6037
BRISQUE                       : 1.0742
PIQE                          : 11.8245
CLIPIQA                       : 0.8387


100%|██████████| 50/50 [00:07<00:00,  6.47it/s]


CLIP Score                    : 11.4054
NIQE                          : 2.9957
BRISQUE                       : 18.1467
PIQE                          : 18.5376
CLIPIQA                       : 0.6453


100%|██████████| 50/50 [00:07<00:00,  6.63it/s]


CLIP Score                    : 25.6432
NIQE                          : 3.1223
BRISQUE                       : -4.3561
PIQE                          : 12.6975
CLIPIQA                       : 0.8659


100%|██████████| 50/50 [00:07<00:00,  6.55it/s]


CLIP Score                    : 25.0661
NIQE                          : 2.8763
BRISQUE                       : 9.8942
PIQE                          : 14.3772
CLIPIQA                       : 0.7927


100%|██████████| 50/50 [00:07<00:00,  6.41it/s]


CLIP Score                    : 21.6758
NIQE                          : 5.3133
BRISQUE                       : 25.1845
PIQE                          : 18.3083
CLIPIQA                       : 0.4368


In [5]:
blip_text_encoder = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device).text_decoder
blip_tokenizer = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
adapter_blip = torch.load('/home/anmilka/different/genai/adapter_blip.pth', weights_only=False)

for prompt in prompts:
    infer_model(pipe, blip_text_encoder, blip_tokenizer, prompt, " ", adapter_blip)

del blip_text_encoder
torch.cuda.empty_cache()

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
100%|██████████| 50/50 [00:07<00:00,  6.66it/s]


CLIP Score                    : 17.4695
NIQE                          : 8.6235
BRISQUE                       : 46.6049
PIQE                          : 52.6965
CLIPIQA                       : 0.7921


100%|██████████| 50/50 [00:07<00:00,  6.67it/s]


CLIP Score                    : 14.1571
NIQE                          : 3.4536
BRISQUE                       : 20.2969
PIQE                          : 45.0938
CLIPIQA                       : 0.7669


100%|██████████| 50/50 [00:07<00:00,  6.68it/s]


CLIP Score                    : 26.9347
NIQE                          : 4.1473
BRISQUE                       : 21.2070
PIQE                          : 33.9716
CLIPIQA                       : 0.7501


100%|██████████| 50/50 [00:07<00:00,  6.66it/s]


CLIP Score                    : 12.0230
NIQE                          : 3.9927
BRISQUE                       : -4.3416
PIQE                          : 15.0649
CLIPIQA                       : 0.7697


100%|██████████| 50/50 [00:07<00:00,  6.64it/s]


CLIP Score                    : 13.7377
NIQE                          : 3.2915
BRISQUE                       : 6.0617
PIQE                          : 26.1599
CLIPIQA                       : 0.8683


In [4]:
qwen_text_encoder = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Instruct-2507",
    torch_dtype=torch.float32,
    trust_remote_code=True
).to(device)
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507", trust_remote_code=True)
adapter_qwen = torch.load('/home/anmilka/different/genai/adapter_qwen.pth', weights_only=False)

for prompt in prompts:
    infer_model(pipe, qwen_text_encoder, qwen_tokenizer, prompt, " ", adapter_qwen)

del qwen_text_encoder
torch.cuda.empty_cache()

`torch_dtype` is deprecated! Use `dtype` instead!
100%|██████████| 50/50 [00:07<00:00,  6.44it/s]
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/anmilka/miniconda3/envs/common/lib/python3.10/site-packages/pyiqa/matlab_utils/functions.py:173: UserWarning: cov(): degrees of freedom is <= 0. Correction should be strictly less than the number of observations. (Triggered internally at /pytorch/aten/src/ATen/native/Correlation.cpp:116.)
  return torch.cov(tensor, correction=correction)
/home/anmilka/miniconda3/envs/common/lib/python3.10/site-packages/pyiqa/archs/niqe_arch.py:161: UserWarning: torch.linalg.svd: During SVD computation with the selected cusolver driver, batches 0 failed to converge. A more accurate method will be used to compute the SVD as a fallback. Check doc at https://pytorch.org/docs/stable/

CLIP Score                    : 13.1812
NIQE                          : 6.5464
BRISQUE                       : 88.7856
PIQE                          : 50.5996
CLIPIQA                       : 0.7241


100%|██████████| 50/50 [00:07<00:00,  6.63it/s]


CLIP Score                    : 7.9935
NIQE                          : 5.7304
BRISQUE                       : 40.5129
PIQE                          : 62.1849
CLIPIQA                       : 0.4644


100%|██████████| 50/50 [00:07<00:00,  6.63it/s]


CLIP Score                    : 13.7106
NIQE                          : 6.7913
BRISQUE                       : 60.3168
PIQE                          : 73.5584
CLIPIQA                       : 0.6487


100%|██████████| 50/50 [00:07<00:00,  6.64it/s]
